# 02 - Time-based medal prediction

This notebook runs a chronological backtest and a retrospective Tokyo 2020 forecast. The momentum-adjusted Ridge model uses the NOC's previous observed medal total, previous delegation size, host status, and Games year. A two-stage Logistic Regression + Poisson model and a last-observed-Games baseline are retained as comparators. World Bank features use the year before each Games. 2012 is the validation/model-selection Games; 2016 is the final historical holdout.

In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT.resolve()))

from src.data_preparation import build_modeling_table
from src.modeling import evaluate_2020_forecast, run_2020_forecast, run_backtest
from src.visualization import create_figures

PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'

if not (PROCESSED_DIR / 'country_year_modeling_table.csv').exists():
    _, preparation_diagnostics = build_modeling_table(PROJECT_ROOT / 'data' / 'raw', PROCESSED_DIR)
    print(preparation_diagnostics)

metrics, predictions = run_backtest(PROCESSED_DIR)
forecast_2020 = run_2020_forecast(PROCESSED_DIR)
actual_metrics_2020, comparison_2020 = evaluate_2020_forecast(PROCESSED_DIR)
display(metrics)
display(actual_metrics_2020)
figure_paths = create_figures(PROJECT_ROOT)
print('Saved figures:', [path.relative_to(PROJECT_ROOT).as_posix() for path in figure_paths])

,Model,Metric,Value,Evaluation_Year,Evaluation_Role
0,Two-stage classifier + Poisson regressor,MAE total medals,2.753516,NaN,NaN
1,Two-stage classifier + Poisson regressor,RMSE total medals,5.117756,NaN,NaN
2,Last-observed-Games baseline,MAE total medals,1.285012,NaN,NaN
3,Last-observed-Games baseline,RMSE total medals,3.048744,NaN,NaN
4,Medal-winner classifier,Accuracy,0.877150,NaN,NaN
5,Medal-winner classifier,Balanced accuracy,0.874559,NaN,NaN
6,Medal-winner classifier,Precision,0.848837,NaN,NaN
7,Medal-winner classifier,Recall,0.858824,NaN,NaN
8,Medal-winner classifier,F1,0.853801,NaN,NaN
9,Medal-winner classifier,ROC AUC,0.934227,NaN,NaN


,Model,Metric,Value
0,Momentum-adjusted Ridge,MAE total medals,1.555574
1,Momentum-adjusted Ridge,RMSE total medals,3.558346
2,Two-stage classifier + Poisson,MAE total medals,2.686648
3,Two-stage classifier + Poisson,RMSE total medals,6.260613
4,Last-observed-Games baseline,MAE total medals,1.682927
5,Last-observed-Games baseline,RMSE total medals,3.672575
6,Momentum-adjusted Ridge,Residual standard deviation,3.507551
7,Two-stage classifier + Poisson,Residual standard deviation,6.243926
8,Last-observed-Games baseline,Residual standard deviation,3.633881
9,Momentum-adjusted Ridge,Top-10 medal-country MSE,173.303875


Saved figures: ['outputs/figures/tokyo_2020_forecast_vs_actual.png', 'outputs/figures/rio_2016_holdout_predictions.png']


In [2]:
top_predictions = (
    predictions.sort_values(['Year', 'Expected_Total_Prediction'], ascending=[True, False])
    .groupby('Year', group_keys=False)
    .head(10)
)
display(top_predictions)
display(forecast_2020.head(15))
display(comparison_2020[['NOC', 'Predicted_Total', 'Actual_Total', 'Forecast_Status']].head(20))

,Year,NOC,Olympic_Region,WB_Country_Code,Actual_Total,Actual_Won_Any_Medal,Predicted_Win_Probability,Predicted_Winner_At_0_5,Predicted_Medals_If_Winner,Expected_Total_Prediction,Previous_Observed_Total_Baseline,Rounded_Total_Prediction
133,2012,USA,USA,USA,103,1,1.000000,1,128.031514,128.031514,110.0,128
1,2012,CHN,China,CHN,88,1,1.000000,1,112.978426,112.978426,100.0,113
359,2012,GBR,UK,GBR,65,1,1.000000,1,46.397655,46.397655,48.0,46
33,2012,RUS,Russia,RUS,82,1,1.000000,1,44.222095,44.222095,72.0,44
448,2012,GER,Germany,DEU,44,1,1.000000,1,29.180986,29.180986,41.0,29
15,2012,FRA,France,FRA,35,1,1.000000,1,26.961235,26.961235,41.0,27
241,2012,JPN,Japan,JPN,38,1,1.000000,1,26.517928,26.517927,25.0,27
127,2012,AUS,Australia,AUS,35,1,1.000000,1,22.402027,22.402027,46.0,22
55,2012,ITA,Italy,ITA,28,1,1.000000,1,20.672960,20.672959,27.0,21
475,2012,BRA,Brazil,BRA,17,1,0.999995,1,20.425082,20.424982,16.0,20


,Year,NOC,Olympic_Region,WB_Country_Code,WB_Country_Name,Feature_Year,GDP_USD,Population,Land_Area_Sq_Km,Previous_Observed_Total,...,Predicted_Win_Probability,Predicted_Winner_At_Tuned_Threshold,Predicted_Medals_If_Winner,Two_Stage_Expected_Total_Prediction,Ridge_Count_Prediction,Expected_Total_Prediction,Rounded_Total_Prediction,Ridge_Alpha_Selected_On_2012,Momentum_Blend_Weight_Selected_On_2012,Classifier_Threshold
195,2020,USA,USA,USA,United States,2019,2.153998e+13,3.302262e+08,9.147420e+06,121,...,1.0,1,157.791567,157.791567,109.701713,109.701713,110,1.0,1.0,0.56
38,2020,CHN,China,CHN,China,2019,1.456017e+13,1.407745e+09,9.388210e+06,70,...,1.0,1,74.180797,74.180797,64.376515,64.376515,64,1.0,1.0,0.56
67,2020,GBR,UK,GBR,United Kingdom,2019,2.875710e+12,6.662800e+07,2.419300e+05,67,...,1.0,1,38.327567,38.327567,61.388920,61.388920,61,1.0,1.0,0.56
95,2020,JPN,Japan,JPN,Japan,2019,5.245755e+12,1.266330e+08,3.645000e+05,41,...,1.0,1,45.822762,45.822762,58.963924,58.963924,59,1.0,1.0,0.56
156,2020,RUS,Russia,RUS,Russian Federation,2019,1.693115e+12,1.454533e+08,1.637687e+07,56,...,1.0,1,33.482462,33.482462,50.972877,50.972877,51,1.0,1.0,0.56
71,2020,GER,Germany,DEU,Germany,2019,3.959895e+12,8.309296e+07,3.493900e+05,42,...,1.0,1,27.653105,27.653105,41.370228,41.370228,41,1.0,1.0,0.56
63,2020,FRA,France,FRA,France,2019,2.722794e+12,6.738206e+07,5.389500e+05,42,...,1.0,1,24.663291,24.663291,40.958454,40.958454,41,1.0,1.0,0.56
10,2020,AUS,Australia,AUS,Australia,2019,1.398350e+12,2.533483e+07,7.692020e+06,29,...,1.0,1,17.272142,17.272142,30.514526,30.514526,31,1.0,1.0,0.56
91,2020,ITA,Italy,ITA,Italy,2019,2.019607e+12,5.972908e+07,2.957170e+05,28,...,1.0,1,18.431867,18.431867,27.919078,27.919078,28,1.0,1.0,0.56
33,2020,CAN,Canada,CAN,Canada,2019,1.743729e+12,3.761850e+07,8.965590e+06,22,...,1.0,1,15.996726,15.996726,22.909972,22.909972,23,1.0,1.0,0.56


,NOC,Predicted_Total,Actual_Total,Forecast_Status
196,USA,109.701713,113,forecasted and won medals
38,CHN,64.376515,88,forecasted and won medals
67,GBR,61.388920,65,forecasted and won medals
95,JPN,58.963924,58,forecasted and won medals
157,RUS,50.972877,71,forecasted and won medals
71,GER,41.370228,37,forecasted and won medals
63,FRA,40.958454,33,forecasted and won medals
10,AUS,30.514526,46,forecasted and won medals
91,ITA,27.919078,40,forecasted and won medals
33,CAN,22.909972,24,forecasted and won medals


## Interpretation notes

Read 2012 as a validation score, not as an independent final test: it is used to select the model settings. The 2016 score is the chronological final holdout. Missing World Bank values remain missing and are imputed inside each estimator. The retrospective 2020 comparison uses the Tokyo medal-total file in data/raw; ROC is mapped to historical NOC RUS. Forecast candidates come from 2016, so actual medal winners outside that candidate list are included with a zero forecast. The Tokyo medal file lists medal winners only; it cannot distinguish non-participation from participation without a medal, so treat the full-table Tokyo metrics with that limitation in mind.